# Tempo Run 2025 — Push model lên Hugging Face Hub

Chỉ chạy khi kết quả evaluation tốt. Có 2 lựa chọn:
- Push **LoRA adapter** (chỉ ~50-100MB) — người dùng tự load base + adapter
- Push **merged model** (~1.2GB cho 0.6B bf16) — load thẳng

In [ ]:
%cd /content/finetune_1B_MCQ_VN
from temprun.utils import load_env, repo_root
load_env(repo_root() / ".env")
import os
assert os.environ.get("HF_TOKEN"), "HF_TOKEN missing"
assert os.environ.get("HF_REPO"), "HF_REPO missing"

In [ ]:
# Lựa chọn: push gì?
MODE = "lora"  # "lora" | "merged"
RUN  = "qlora_qwen3_0_6b"

In [ ]:
import os
from huggingface_hub import HfApi, create_repo
from temprun.utils import load_env, repo_root
load_env(repo_root() / ".env")

REPO = os.environ["HF_REPO"]
api = HfApi(token=os.environ["HF_TOKEN"])
create_repo(REPO, token=os.environ["HF_TOKEN"], exist_ok=True, private=False)
print(f"HF repo ready: https://huggingface.co/{REPO}")

In [ ]:
if MODE == "lora":
    # Push LoRA adapter (chỉ cần file trong artifacts/<run>/)
    api.upload_folder(
        folder_path=f"artifacts/{RUN}",
        repo_id=REPO,
        commit_message=f"Upload LoRA adapter from {RUN}",
        ignore_patterns=["*.bin", "*.safetensors", "global_step/*", "runs/*", "eval_details*"],
    )
elif MODE == "merged":
    # Merge + push model đầy đủ
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer
    import torch, shutil
    base = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B", torch_dtype=torch.bfloat16, device_map="cpu")
    model = PeftModel.from_pretrained(base, f"artifacts/{RUN}")
    model = model.merge_and_unload()
    tok = AutoTokenizer.from_pretrained(f"artifacts/{RUN}")
    out = f"artifacts/{RUN}_merged"
    os.makedirs(out, exist_ok=True)
    model.save_pretrained(out, safe_serialization=True, max_shard_size="500MB")
    tok.save_pretrained(out)
    api.upload_folder(folder_path=out, repo_id=REPO, commit_message=f"Upload merged {RUN}")
print(f"Pushed to https://huggingface.co/{REPO}")